In [1]:
from regional_mom6 import regridding as rgd
import json
import xarray as xr
import numpy as np
from pathlib import Path
from CrocoDash.vgrid import VGrid

In [28]:
with open("/glade/u/home/manishrv/documents/croc/dev/bgc/cesm_downscaling/list_of_physicals.json")as f:
    tracers = json.load(f)["tracers"]
regridded_folder = Path("/glade/u/home/manishrv/documents/croc/dev/bgc/cesm_downscaling/regridded_interpolated_filled_data")
out_folder = Path("/glade/u/home/manishrv/documents/croc/dev/bgc/cesm_downscaling/processed_interpolated_filled_data")
old_ic_eta = xr.open_dataset("/glade/derecho/scratch/manishrv/croc_input/marbl.bio.croc.4/ocnice/init_eta.nc")
old_ic_tracers = xr.open_dataset("/glade/derecho/scratch/manishrv/croc_input/marbl.bio.croc.4/ocnice/init_tracers.nc")
old_ic_vel = xr.open_dataset("/glade/derecho/scratch/manishrv/croc_input/marbl.bio.croc.4/ocnice/init_vel.nc")
phys_new_files = {
}

for tracer in tracers:
    phys_new_files[tracer] = xr.open_dataset(regridded_folder / f"{tracer}_ic_regridded_with_fills.nc")
hgrid = xr.open_dataset("/glade/derecho/scratch/manishrv/croc_input/nwa_in_crocesm/ocnice/ocean_hgrid.nc")
u_points = rgd.get_hgrid_arakawa_c_points(hgrid, "u")
v_points = rgd.get_hgrid_arakawa_c_points(hgrid, "v")
t_points = rgd.get_hgrid_arakawa_c_points(hgrid, "t")  
vgrid = VGrid.from_file("/glade/derecho/scratch/manishrv/croc_input/marbl.bio.croc.4/ocnice/ocean_vgrid_nwa4_ca99a7.nc")
cell_centers = np.cumsum(vgrid.dz) - 0.5 * vgrid.dz



In [22]:
def multidirectional_fill_xarray(da, z_dim, y_dim, x_dim):
    # Vertical fill (down then up)
    if z_dim is not None:
        da = da.ffill(dim=z_dim)
        da = da.bfill(dim=z_dim)
    # Horizontal fill (left then right) along x
    da = da.bfill(dim=x_dim)
    da = da.ffill(dim=x_dim)

    da = da.ffill(dim=y_dim)
    da = da.bfill(dim=y_dim)
    # Final vertical fill (down)
    if z_dim is not None:
        da = da.ffill(dim=z_dim)
        da = da.bfill(dim=z_dim)
    return da

In [33]:
# Tracers
variable_encodings = {
    var: old_ic_tracers[var].encoding for var in old_ic_tracers.data_vars
}

val = phys_new_files["TEMP"]["__xarray_dataarray_variable__"][0].interp({"z_t":cell_centers*100})
val = val.where(val != 0, np.nan)
val = multidirectional_fill_xarray(val, "z_t", "nyp", "nxp")

old_ic_tracers["temp"] = (("zl", "ny", "nx"),val[:,t_points.t_points_y,t_points.t_points_x].values)
val = phys_new_files["SALT"]["__xarray_dataarray_variable__"][0].interp({"z_t":cell_centers*100})
val = val.where(val != 0, np.nan)
val = multidirectional_fill_xarray(val, "z_t", "nyp", "nxp")
old_ic_tracers["salt"] = (("zl", "ny", "nx"),val[:,t_points.t_points_y,t_points.t_points_x].values)
# Step 4: Reapply encodings and attributes
for var, encoding in variable_encodings.items():
    old_ic_tracers[var].encoding.update(encoding)

old_ic_tracers.to_netcdf(out_folder / "init_tracers.nc")


In [34]:
# Velocity
# List of valid encoding keys for netCDF4 backend
variable_encodings = {
    var: old_ic_vel[var].encoding for var in old_ic_vel.data_vars
}

val = phys_new_files["UVEL"]["__xarray_dataarray_variable__"][0].interp({"z_t":cell_centers*100})
val = val.where(val != 0, np.nan)
val = multidirectional_fill_xarray(val, "z_t", "nyp", "nxp") / 100
old_ic_vel["u"] = (("zl", "ny", "nxp"),val[:,u_points.u_points_y,u_points.u_points_x].values)
val = phys_new_files["VVEL"]["__xarray_dataarray_variable__"][0].interp({"z_t":cell_centers*100})
val = val.where(val != 0, np.nan)
val = multidirectional_fill_xarray(val, "z_t", "nyp", "nxp") / 100
old_ic_vel["v"] = (("zl", "nyp", "nx"),val[:,v_points.v_points_y,v_points.v_points_x].values)
for var, encoding in variable_encodings.items():
    old_ic_vel[var].encoding.update(encoding)
old_ic_vel.to_netcdf(out_folder / "init_vel.nc")


In [35]:
# ETA
# List of valid encoding keys for netCDF4 backend
variable_encodings = {
    var: old_ic_eta[var].encoding for var in old_ic_eta.data_vars
}

val = phys_new_files["SSH"]["__xarray_dataarray_variable__"][0]
val = val.where(val != 0, np.nan)
val = multidirectional_fill_xarray(val, z_dim = None, y_dim="nyp", x_dim="nxp")
old_ic_eta["eta_t"] = (( "ny", "nx"),val[t_points.t_points_y,t_points.t_points_x].values)
for var, encoding in variable_encodings.items():
    old_ic_eta[var].encoding.update(encoding)
old_ic_eta.to_netcdf(out_folder / "init_eta.nc")


In [15]:
val

<xarray.DataArray '__xarray_dataarray_variable__' (nyp: 1561, nxp: 1481)>
array([[  15.425262,   15.392029,   15.358437, ...,    5.274397,    5.231397,
           5.188454],
       [  15.414678,   15.390779,   15.351239, ...,    5.221614,    5.176351,
           5.132312],
       [  15.411475,   15.378292,   15.344354, ...,    5.166088,    5.120801,
           5.075556],
       ...,
       [   0.      ,    0.      ,    0.      , ..., -120.789375, -120.648674,
        -120.503   ],
       [   0.      ,    0.      ,    0.      , ..., -120.766884, -120.67271 ,
        -120.57163 ],
       [   0.      ,    0.      ,    0.      , ..., -120.748634, -120.700966,
        -120.643555]], dtype=float32)
Coordinates:
    time     object 2000-02-01 00:00:00
Dimensions without coordinates: nyp, nxp
Attributes:
    regrid_method:  bilinear

In [12]:
phys_new_files["SSH"]["__xarray_dataarray_variable__"][0]

<xarray.DataArray '__xarray_dataarray_variable__' (nyp: 1561, nxp: 1481)>
[2311841 values with dtype=float32]
Coordinates:
    time     object 2000-02-01 00:00:00
Dimensions without coordinates: nyp, nxp
Attributes:
    regrid_method:  bilinear